<a href="https://colab.research.google.com/github/salvavidassince2018-eng/dissertacao/blob/main/codigosprincipaisdirce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Começar montando o drive...

In [ ]:
# prompt: from google.colab import drive  # Montar o Google Drive drive.mount('/content/drive')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

# Caminho para o arquivo CSV
caminho_arquivo = '/content/drive/MyDrive/Cópia de dfSVM.csv'

# Carregar o arquivo CSV em um DataFrame do pandas
df = pd.read_csv(caminho_arquivo)

# Exibir as primeiras linhas do DataFrame para verificar se o carregamento foi bem-sucedido
df.head()


In [ ]:
df.info()

In [ ]:
import pandas as pd

# Supondo que df seja o seu DataFrame
numero_de_linhas = len(df)
print(f'O número de linhas no DataFrame é: {numero_de_linhas}')


codigo completo com agregação e sem lira - atual nrc

In [ ]:
# Instalar bibliotecas (se necessário)
!pip install unidecode
!pip install nltk

# 1. Importar bibliotecas
import pandas as pd
import re
from nltk.tokenize import TreebankWordTokenizer
from nltk.util import ngrams
from unidecode import unidecode
from collections import Counter

# 2. Inicializar o tokenizador
tokenizer = TreebankWordTokenizer()

# 3. Função de pré-processamento
def preprocess(text):
    text = unidecode(str(text).lower())
    text = re.sub(r"http\S+|@\S+|#\S+|[^a-z\s]", " ", text)
    tokens = tokenizer.tokenize(text)
    return tokens

# 4. Carregar e tratar o dicionário NRC
nrc_path = "/content/drive/MyDrive/Portuguese-NRC-EmoLex.txt"  # ajuste o caminho conforme necessário
nrc_df = pd.read_csv(nrc_path, sep='\t')

# 4.1 Definir colunas emocionais
emo_cols = ['anger','anticipation','disgust','fear','joy','negative','positive','sadness','surprise','trust']

# 4.2 Normalizar e remover palavras indesejadas
nrc_df['word'] = nrc_df['Portuguese Word'].apply(lambda x: unidecode(str(x).strip().lower()))
nrc_df = nrc_df[nrc_df['word'] != 'lira']

# 4.3 Preencher NAs com 0 e transformar em inteiro
nrc_df[emo_cols] = nrc_df[emo_cols].fillna(0).astype(int)

# 4.4 Agrupar garantindo que não duplique emoções por palavra (1 por emoção no máximo)
nrc_terms = nrc_df.groupby('word')[emo_cols].max().reset_index().set_index('word')

# 5. Função para gerar n-gramas
def generate_ngrams(tokens, n_max=4):
    all_ngrams = []
    for n in range(1, n_max + 1):
        all_ngrams += [' '.join(grama) for grama in ngrams(tokens, n)]
    return all_ngrams

# 6. Função de classificação com captura de termos
def classificar_emocoes_ngrams_verbose(tokens):
    ngram_list = generate_ngrams(tokens, n_max=4)
    c = Counter()
    encontrados = set()
    for ngram in ngram_list:
        if ngram in nrc_terms.index:
            encontrados.add(ngram)
            emo_row = nrc_terms.loc[ngram]
            for emo in emo_cols:
                if emo_row[emo] == 1:
                    c[emo] += 1  # conta total de emoções, não só presença
    return pd.Series([dict(c), ', '.join(encontrados)], index=['nrc_scores', 'termos_identificados'])

# 7. Aplicar à base
# df = pd.read_csv("SUA_PLANILHA.csv")  # ajuste conforme necessário
df['tokens'] = df['Full Text'].apply(preprocess)

# 8. Aplicar classificação
resultado = df['tokens'].apply(classificar_emocoes_ngrams_verbose)
df = pd.concat([df, resultado], axis=1)

# 9. Expandir colunas de emoções
emo_df = df['nrc_scores'].apply(pd.Series).fillna(0).astype(int)

# 10. Juntar com colunas finais
colunas_finais = ['Date', 'Author', 'Full Text', 'Hashtags', 'Mentioned Authors',
                  'Twitter Retweets', 'Twitter Likes', 'Reach (new)', 'Url', 'termos_identificados']
df_nrc = pd.concat([df[colunas_finais], emo_df], axis=1)

# 11. Visualizar resultado
df_nrc.head()


Base salva - tem que ajustar para salvar no drive e msotra o caminho

In [ ]:
# 12. Salvar base categorizada no Google Drive
output_path = "/content/drive/MyDrive/classificacao_emocional_completa.csv"
df_nrc.to_csv(output_path, index=False)

# 13. Mostrar caminho salvo
print(f"Base salva em: {output_path}")



amostra auditar

In [ ]:
# Amostra aleatória de 3 textos
amostra = df_nrc.sample(4, random_state=42)  # define uma semente para reprodutibilidade

# Exibir apenas as colunas relevantes: texto completo + emoções
colunas_exibir = ['Full Text', 'termos_identificados'] + emo_cols
amostra_resultado = amostra[colunas_exibir]

# Visualizar
amostra_resultado


+ MFD BR

dicionario expandido

In [ ]:
# Instalar bibliotecas necessárias
!pip install unidecode nltk

# Importar
import pandas as pd
import re
from unidecode import unidecode
from nltk.tokenize import TreebankWordTokenizer
from collections import defaultdict

# Caminho do dicionário com prefixos
mfd_path = "/content/drive/MyDrive/1. Mestrado/Dissertação/entrega julho/MFD/mfd_ptbr_alpha.dic"

# 1. Carregar dicionário com asteriscos
prefixo_fundamento = defaultdict(set)
with open(mfd_path, 'r', encoding='utf-8') as f:
    for line in f:
        partes = line.strip().split('\t')
        if len(partes) > 1:
            termo_asterisco = partes[0].strip()
            prefixo = unidecode(termo_asterisco.lower().replace('*', ''))
            for fundamento in partes[1:]:
                prefixo_fundamento[(prefixo, termo_asterisco)].add(fundamento)

# 2. Carregar sua base de tweets
df = pd.read_csv("/content/drive/MyDrive/classificacao_emocional_completa.csv")
tokenizer = TreebankWordTokenizer()

def preprocess(text):
    text = unidecode(str(text).lower())
    text = re.sub(r"http\S+|@\S+|#\S+|[^a-z\s]", " ", text)
    return tokenizer.tokenize(text)

# 3. Construir vocabulário único
vocabulario = set()
for texto in df['Full Text'].dropna():
    vocabulario.update(preprocess(texto))

# 4. Expandir o dicionário
dicionario_expandido = []
for (prefixo, original_com_asterisco), fundamentos in prefixo_fundamento.items():
    palavras = [w for w in vocabulario if w.startswith(prefixo)]
    for palavra in palavras:
        for fundamento in fundamentos:
            dicionario_expandido.append((palavra, fundamento, original_com_asterisco))

# 5. Gerar DataFrame e salvar
df_expandido = pd.DataFrame(dicionario_expandido, columns=["palavra", "fundamento", "prefixo_original"])
df_expandido = df_expandido.drop_duplicates().sort_values(by=["palavra", "fundamento"])

# 6. Contagens
termos_unicos = df_expandido["palavra"].nunique()
linhas_total = len(df_expandido)

print(f"✅ Total de palavras únicas no novo dicionário: {termos_unicos}")
print(f"✅ Total de combinações palavra/fundamento: {linhas_total}")

# 7. Salvar
df_expandido.to_csv("/content/mfd_expandido_com_origem.csv", index=False)
print("📁 Arquivo salvo: /content/mfd_expandido_com_origem.csv")


In [ ]:
df_expandido = pd.read_csv("/content/mfd_expandido_com_origem.csv")


In [ ]:
# 1. Montar o Google Drive (faça isso só uma vez por sessão)
from google.colab import drive
drive.mount('/content/drive')

# 2. Definir caminho de saída no seu Drive (ajuste se quiser salvar em outra pasta)
caminho_saida = "/content/drive/MyDrive/mfd_expandido_com_origem_salvo.xlsx"

# 3. Salvar o DataFrame como Excel
df_expandido.to_excel(caminho_saida, index=False)

# 4. Mostrar o caminho salvo
print(f"✅ Arquivo salvo em: {caminho_saida}")


In [ ]:
# Função de auditoria: detecta palavras com múltiplos fundamentos
def auditar_ambiguidade(df_dicionario):
    # Agrupar e contar quantos fundamentos por palavra
    ambiguidade = df_dicionario.groupby("palavra")["fundamento"].nunique().reset_index()
    ambiguidade.columns = ["palavra", "n_fundamentos"]

    # Filtrar palavras ambíguas (2 ou mais fundamentos)
    ambiguas = ambiguidade[ambiguidade["n_fundamentos"] > 1].sort_values(by="n_fundamentos", ascending=False)

    # Juntar com os fundamentos associados para inspeção
    fundamentos_associados = df_dicionario.groupby("palavra")["fundamento"].apply(list).reset_index()

    resultado = pd.merge(ambiguas, fundamentos_associados, on="palavra")
    return resultado



In [ ]:
df_ambiguidade = auditar_ambiguidade(df_expandido)
df_ambiguidade.head(10)  # visualizar as 10 mais ambíguas


Classificação com o dicionario expandido

In [ ]:
# Instalar bibliotecas necessárias
!pip install unidecode nltk openpyxl tqdm

# Importar bibliotecas
import pandas as pd
import re
from unidecode import unidecode
from nltk.tokenize import TreebankWordTokenizer
from collections import Counter
from tqdm.notebook import tqdm
tqdm.pandas()

# Caminhos dos arquivos
df_path = "/content/drive/MyDrive/classificacao_emocional_completa.csv"
mfd_path = "/content/drive/MyDrive/mfd_expandido_com_origem_salvo.xlsx"

# Inicializar tokenizador
tokenizer = TreebankWordTokenizer()

# Função de pré-processamento
def preprocess(text):
    text = unidecode(str(text).lower())
    text = re.sub(r"http\S+|@\S+|#\S+|[^a-z\s]", " ", text)
    return tokenizer.tokenize(text)

# Carregar dicionário expandido
mfd_df = pd.read_excel(mfd_path)
mfd_df['fundamento'] = mfd_df['fundamento'].astype(str)
mfd_df['palavra'] = mfd_df['palavra'].astype(str)

# Palavras proibidas
palavras_proibidas = ["paises", "leia", "leitura", "alias"]

# Remover palavras indesejadas do dicionário
mfd_df = mfd_df[~mfd_df['palavra'].isin(palavras_proibidas)]

# Transformar dicionário em matriz binária
mfd_pivot = (
    mfd_df.assign(valor=1)
    .pivot_table(index="palavra", columns="fundamento", values="valor", aggfunc="max")
    .fillna(0)
    .astype(int)
)
mfd_pivot.columns.name = None
mfd_pivot = mfd_pivot.reset_index().set_index('palavra')
mfd_fundamentos = list(mfd_pivot.columns)

# Nomes dos fundamentos em inglês
fundamento_nomes = {
    '1': 'HarmVirtue',
    '2': 'HarmVice',
    '3': 'FairnessVirtue',
    '4': 'FairnessVice',
    '5': 'IngroupVirtue',
    '6': 'IngroupVice',
    '7': 'AuthorityVirtue',
    '8': 'AuthorityVice',
    '9': 'PurityVirtue',
    '10': 'PurityVice',
    '11': 'MoralityGeneral'
}
fundamento_nomes = {str(k): v for k, v in fundamento_nomes.items()}

# Função de classificação com filtro
def classificar_fundamentos(row):
    texto_original = row['Full Text']
    if any(bad_word in texto_original.lower() for bad_word in palavras_proibidas):
        return pd.Series([{}, ''], index=['mfd_scores', 'palavras_mfd'])

    tokens = row['tokens']
    c = Counter()
    encontrados = set()
    for token in set(tokens):
        if token in mfd_pivot.index:
            encontrados.add(token)
            for fund in mfd_fundamentos:
                if mfd_pivot.loc[token, fund] == 1:
                    c[fund] += 1
    return pd.Series([dict(c), ', '.join(sorted(encontrados))], index=['mfd_scores', 'palavras_mfd'])

# Carregar base completa
df = pd.read_csv(df_path)

# Tokenizar com barra de progresso
df['tokens'] = df['Full Text'].progress_apply(preprocess)

# Classificar com barra de progresso
resultado = df.progress_apply(classificar_fundamentos, axis=1)
df = pd.concat([df, resultado], axis=1)

# Expandir colunas dos fundamentos
fundamentos_df = df['mfd_scores'].apply(pd.Series).fillna(0).astype(int)
fundamentos_df.columns = [fundamento_nomes.get(str(col), col) for col in fundamentos_df.columns]

# Juntar tudo
df_final = pd.concat([df.drop(columns=['tokens', 'mfd_scores']), fundamentos_df], axis=1)

# Salvar como Excel no Google Drive
output_path = "/content/drive/MyDrive/base_nrc_mfd.xlsx"
df_final.to_excel(output_path, index=False)

# Mostrar caminho salvo
print(f"✅ Arquivo salvo com sucesso em: {output_path}")


Base completa- NRC e MFD colunas de preponderante/s

In [ ]:
# Instalar se necessário
!pip install openpyxl

# Importar bibliotecas
import pandas as pd

# 1. Caminho da planilha original
caminho_entrada = "/content/drive/MyDrive/base_nrc_mfd.xlsx"
df = pd.read_excel(caminho_entrada)

# 2. Identificar colunas do NRC e MFD (conforme imagem)
nrc_cols = ['positive', 'anticipation', 'trust', 'anger', 'disgust',
            'negative', 'sadness', 'joy', 'surprise', 'fear']

# Padrão: colunas MFD estão da coluna V (IngroupVirtue) até AF (PurityVice)
mfd_cols = df.columns[df.columns.get_loc("IngroupVirtue"):df.columns.get_loc("PurityVice")+1].tolist()

# 3. Função para obter valor predominante com empate
def obter_predominante(row, colunas):
    valores = row[colunas]
    max_valor = valores.max()
    if max_valor == 0:
        return ''
    return ', '.join(valores[valores == max_valor].index.tolist())

# 4. Aplicar para criar colunas novas
df['nrc_predominante'] = df.apply(lambda row: obter_predominante(row, nrc_cols), axis=1)
df['mfd_predominante'] = df.apply(lambda row: obter_predominante(row, mfd_cols), axis=1)

# 5. Salvar no Drive
caminho_saida = "/content/drive/MyDrive/base_nrc_mfd_com_predominantes.xlsx"
df.to_excel(caminho_saida, index=False)

print(f"✅ Base com colunas 'nrc_predominante' e 'mfd_predominante' salva em:\n{caminho_saida}")



sem retweets e na base classificada morais etc -- 17.07

In [ ]:
import pandas as pd
import re

# Caminho para o arquivo CSV
caminho = "/content/drive/MyDrive/2base_nrc_mfd_completa_predominantes.xlsx"
df = pd.read_excel(caminho)


# Remover linhas onde o 'Full Text' começa com 'RT @' (retweets)
df_sem_retweets = df[~df['Full Text'].str.startswith('RT @')]

# Exibir a quantidade de linhas após a remoção dos retweets
quantidade_linhas_sem_retweets = df_sem_retweets.shape[0]
print(f'A quantidade de linhas após a remoção dos retweets é: {quantidade_linhas_sem_retweets}')

# Definir as listas de palavras-chave para cada categoria
a_favor = [
    'PL 2630 é urgente.', 'A PL 2630 tá cada vez mais imprescindível!', '#RegulaJá', 'O PL 2630 precisa ser aprovado',
    'Sim PL 2630!', 'mostra a necessidade de regulamentação das plataformas digitais', 'justifica a PL 2630',
    'dá razão para existência do PL 2630', 'TOTALMENTE A FAVOR do PL 2630', '2630 CONTRA FAKE E HATE', 'Todo apoio ao PL 2630',
    'PL 2630/20 JÁ', 'PL 2630 JÁ', 'PL 2630 precisa ser aprovada', 'prova a necessidade do PL 2630 ser aprovado',
    'é necessário a PL 2630', 'precisamos regular as redes', 'urgência já', 'regula já', '#urgênciajá', '#regulajá',
    'urgência já pl 2630!', 'devemos regular as plataformas', 'regulação é importante', 'pl 2630 sim', 'pl2630sim', 'pl2630já',
    'aprovapl2630', 'pl 2630 já', '#DireitoAutoralSim', 'pl 2630 pelas crianças', 'PL 2630 pelas crianças!', 'PL 2630 SIM',
    'PL 2630 JÁ', 'PL 2630 PELAS CRIANÇAS', 'necessidade da regulação', 'responsabilização das plataformas'
]

contra = [
    'NÃO AO PL 2630', 'PL 2630 NÃO', 'CONTRA a PL DA CENSURA', 'vexame do PL 2630', 'NÃO ao projeto da Censura!!!',
    'PL 2630 é um retrocesso', '#PL 2630Não', 'NÃO ao PL das FakeNews', 'NÃO à censura', 'NÂO ao PL 2630',
    'Não pela PL 2630', '#PLdaCensuraNão', 'Não aceite o PL 2630!', 'Não PL 2630', 'PL2630NÃO!!', '#naoaoPL2630',
    'Diga NÃO à censura!', '#NAOACENSURA', 'CENCURA NÃO PL 2630', 'Não ao PL 2630/20!', 'Não a PL 2630', '#naoapl2630',
    '#CensuraNão', '#pl2630não', 'diganão ao pl 2630!', 'ñ a pl 2630', 'não ao pl da censura!', 'censuranao', 'pldacensuranao'
]

# Função para classificar cada publicação, buscando palavras inteiras
def classificar_categoria(texto):
    texto = texto.lower()  # Tornar o texto minúsculo para facilitar a comparação
    # Verificar se alguma palavra-chave de "a favor" está presente
    if any(re.search(r'\b' + re.escape(palavra.lower()) + r'\b', texto) for palavra in a_favor):
        return 'A Favor'
    # Verificar se alguma palavra-chave de "contra" está presente
    elif any(re.search(r'\b' + re.escape(palavra.lower()) + r'\b', texto) for palavra in contra):
        return 'Contra'
    # Se não encontrar nenhuma das palavras-chave, classificar como "Neutro"
    else:
        return 'Neutro'

# Aplicar a função de classificação à coluna 'Full Text'
df_sem_retweets['Categoria'] = df_sem_retweets['Full Text'].apply(classificar_categoria)

# Exibir as primeiras linhas do DataFrame com as categorias
print(df_sem_retweets[['Author', 'Date', 'Full Text', 'Categoria']].head())

# Contar os totais de cada categoria
totais_categoria = df_sem_retweets['Categoria'].value_counts()
print("\nTotais de cada categoria:")
print(totais_categoria)

# Salvar o DataFrame com as categorias em formato CSV no Google Drive
caminho_destino_categoria_csv = '/content/drive/MyDrive/df_posicionamentos_sem_retweets.csv'
df_sem_retweets.to_csv(caminho_destino_categoria_csv, index=False)

# Exibir o caminho onde o arquivo foi salvo
print(f'O arquivo com as categorias foi salvo em: {caminho_destino_categoria_csv}')


### **codigo para preparar o texto para o IRAMUTEQ - tirei emoji tbm

In [ ]:
# 1. Instalar bibliotecas
!pip install -q pandas openpyxl emoji unidecode

# 2. Importar bibliotecas
import pandas as pd
import re
import emoji
from unidecode import unidecode
from google.colab import drive, files

# 3. Montar Google Drive
drive.mount('/content/drive')

# 4. Parâmetros
caminho = '/content/drive/MyDrive/df_afavor_contra.xlsx'
data_inicial = '2023-04-30'
data_final = '2023-05-02'

# 5. Ler Excel e filtrar por data
df = pd.read_excel(caminho)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df[(df['Date'] >= data_inicial) & (df['Date'] <= data_final)].copy()

# 6. Funções de limpeza
def remover_emojis(texto):
    return emoji.replace_emoji(str(texto), replace='')

def limpar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r"http\S+|www\S+|@\S+|#\S+", " ", texto)
    texto = re.sub(r"\||:|\*|\"|\?|<|>|\||\$|\-|'|%", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# 7. Aplicar limpeza e formatação
df['texto_limpo'] = df['Full Text'].apply(remover_emojis).apply(limpar_texto)
df = df[df['texto_limpo'].str.split().str.len() >= 5].copy()
df['texto_limpo'] = df['texto_limpo'].apply(lambda x: x if x.endswith(('.', '!', '?')) else x + '.')

# 8. Variáveis auxiliares
df['nome'] = 'nome_' + df['Author'].astype(str).apply(unidecode).str.replace(r'\s+', '_', regex=True)
df['data_formatada'] = df['Date'].dt.strftime('data_%Y_%m')
df['pos_cat'] = 'posicionamento_' + df['posicionamento'].astype(str).str.lower().str.replace(' ', '_')

# 9. Agrupar por autor + posicionamento + data
agrupado = df.groupby(['Author', 'nome', 'data_formatada', 'pos_cat'])['texto_limpo'] \
             .apply(lambda x: ' '.join(x)).reset_index()

# 10. Gerar cabeçalhos e textos
linhas_formatadas = []
for i, linha in agrupado.iterrows():
    id_str = f"id_{i+1}"
    nome = linha['nome']
    data = linha['data_formatada']
    pos = linha['pos_cat']
    texto = linha['texto_limpo']

    cabecalho = f"**** *{id_str} *{nome} *{data} *{pos}\n"
    linhas_formatadas.append(f"{cabecalho}{texto}\n")

# 11. Salvar .txt
nome_saida = "/content/iramuteq_afavor_contra_concat_por_autor.txt"
with open(nome_saida, "w", encoding="utf-8") as f:
    f.writelines(linhas_formatadas)

# 12. Download
files.download(nome_saida)
